# Install Dependencies

In [ ]:
!pip install -q openai

# Load Modules

In [ ]:
import pandas as pd
import json
from openai import OpenAI
from sklearn.model_selection import train_test_split
import re

client = OpenAI(api_key='API_KEY')

# Define Functions

In [ ]:
def format_prompt(data):

    """

    Background: This function helps format data into a list of dicts into the required shape for fine tuning

    Params:
    data (list): list of dicts

    Returns:
    training_data_list (list): a list in the proper format for converting to jsonl

    """

    training_data_list = []

    for x in data:

        updated_data = {
            "messages": [
                {
                    "role": "system",
                    "content": x['system_message']
                },
                {
                    "role": "user",
                    "content": x['user_content']
                }
            ]
        }

        training_data_list.append(updated_data)

    print(training_data_list)

    return training_data_list

In [ ]:
def convert_to_jsonl_and_save(data_list, filename):

    """
    Background:
    This function converts the data_list provided into a jsonl file

    Params:
    data_list (list): a list of a dict ready to convert to jsonl
    filename (str): the name of the filename we want to convert

    """

    with open(filename, 'w') as file:
        for data_dict in data_list:
            json_str = json.dumps(data_dict)  # Convert dictionary to JSON string
            file.write(json_str + '\n')  # Write to file with a newline

    print(f"✅ Data successfully written to {filename}")

In [ ]:
def predict(test, model):
  response = client.chat.completions.create(
      model = model,
      messages= test,
      temperature = 0.2,
      max_tokens= 20
  )
  return response.choices[0].message.content

In [ ]:
def store_predictions(test_df, model, test_data):
  print("fine tuned model id is :", model)
  test_df['Prediction']= None

  for index, row in test_df.iterrows():
    test_message = test_data[index]['messages']
    prediction_result = predict(test_message, model)
    test_df.at[index, 'Prediction'] = prediction_result

  test_df.to_csv("predictions.csv")

# Load Model

In [ ]:
model = 'gpt-4.1-2025-04-14'

# Load Data

In [ ]:
data = pd.read_excel('sampled_data.xlsx')

# Zero Shot

## Prepare Prompt

In [ ]:
data['system_message'] = '''You are a language model tasked with classifying Arabic newspaper articles into one of the predefined editorial categories based solely on the article's main topic and content. Carefully read each article and assign only one of the following categories that best reflects its primary subject matter:
1. Culture
2. Finance
3. Medical
4. Politics
5. Religion
6. Sports
7. Tech
'''
data['user_content'] = 'Article:\n' + data['content'] + '\n Predicted Category: '

data2 = data
data = data[['system_message','user_content']]

In [ ]:
data.to_excel('NC-ZeroShot-gpt4-1.xlsx')

In [ ]:
# Convert to JSON String
data = data.to_json(orient='records')

# parse JSON string and convert it into a list of python dictionaries
datadict = json.loads(data)

## Format Prompt

In [ ]:
# format dict into shape for fine tuning
zero_shot_data = format_prompt(datadict)

In [ ]:
# Save as jsonl
convert_to_jsonl_and_save(zero_shot_data, 'NC-ZeroShot.jsonl')

✅ Data successfully written to NC-ZeroShot.jsonl


## Predict

In [ ]:
response = client.chat.completions.create(
      model = model,
      messages= zero_shot_data[66]['messages'],
      temperature = 0.2,
      max_tokens= 20
  )

response.choices[0].message.content

'Finance'

In [ ]:
y_true = data2['subdirectory'].values

In [ ]:
zs = pd.DataFrame()

zs['text'] = data2['content']
zs['Label'] = y_true
zs = zs.reset_index(drop=True)
zs.head()

,text,Label
0,استقبل الوسط المسرحي الإماراتي فوز الإمارات بر...,Culture
1,استضاف مركز الشارقة لفن الخط العربي والزخرفة م...,Culture
2,باسمة يونس قد يبدو العنوان اسماً لرواية؛ لكنه ...,Culture
3,أبوظبي: «الخليج» أكد عدد من الخبراء والمسؤولين...,Culture
4,يمكن القول باطمئنان أن شهر رمضان المبارك هو شه...,Culture


In [ ]:
store_predictions(zs, model, zero_shot_data)

fine tuned model id is : gpt-4.1-2025-04-14


In [ ]:
pred = pd.read_csv('NC-GPT41-ZeroShot-predictions-02Temp.csv')

In [ ]:
pred['Prediction'].value_counts()

,count
Prediction,
Finance,122
Politics,113
Sports,112
Medical,109
Tech,68
Culture,65
Predicted Category: Religion,63
Category: Tech,60
Predicted Category: Politics,46


In [ ]:
pred['Prediction'].isnull().sum()

np.int64(0)

In [ ]:
pred['Prediction'].shape[0]

1000

In [ ]:
import numpy as np

preds = []
for answer in pred['Prediction']:
    if (
          "الرياضة" in answer
          or "Sports" in answer
          or "sports" in answer
          or "sport" in answer
          or "Sport" in answer
      ):
      preds.append("Sports")

    elif (
        "الصحة" in answer
          or "الطب" in answer
          or "Medical" in answer
          or "medical" in answer
        ):
      preds.append("Medical")

    elif (
        "الثقافة" in answer
          or "Culture" in answer
          or "culture" in answer
        ):
      preds.append("Culture")

    elif (
        "المال" in answer
        or "Finance" in answer
        or "finance" in answer
        ):
      preds.append("Finance")

    elif (
        "السياسة" in answer
        or "Politics" in answer
        or "politics" in answer
        ):
      preds.append("Politics")

    elif (
        "الدين" in answer
        or "Religion" in answer
        or "religion" in answer
        ):
      preds.append("Religion")

    elif (
        "التكنولوجيا" in answer
        or "Tech" in answer
        or "Technology" in answer
        or "tech" in answer
        or "technology" in answer
        ):
      preds.append("Tech")

    else:
       preds.append("None")

In [ ]:
np.unique(preds)

array(['Culture', 'Finance', 'Medical', 'Politics', 'Religion', 'Sports',
       'Tech'], dtype='<U8')

In [ ]:
from sklearn.metrics import (f1_score,
                             precision_score,
                             recall_score,
                             classification_report,
                             confusion_matrix)

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score

labels = ['Culture', 'Finance', 'Medical', 'Politics',
          'Religion', 'Sports', 'Tech']
mapping = {'Culture':0, 'Finance':1, 'Medical':2, 'Politics':3,
           'Religion':4, 'Sports':5, 'Tech':6}
def map_func(x):
    return mapping.get(x, 1)

y_true = np.vectorize(map_func)(zs.Label)
y_pred = np.vectorize(map_func)(preds)

# Calculate accuracy
accuracy = accuracy_score(y_true=y_true, y_pred=y_pred)
print(f'Accuracy: {accuracy:.3f}')

# Generate accuracy report
unique_labels = set(y_true)  # Get unique labels

f1t = f1_score(y_true=y_true, y_pred=y_pred, average = 'weighted')
print('\nf1_score: ', f1t)

prec = precision_score(y_true=y_true, y_pred=y_pred, average = 'weighted')
print('\precision: ', prec)

recall = recall_score(y_true=y_true, y_pred=y_pred, average = 'weighted')
print('\recall: ', recall)

# Generate classification report
class_report = classification_report(y_true=y_true,
                                     y_pred=y_pred,
                                     digits = 4,
                                     target_names=labels)
print('\nClassification Report:')
print(class_report)

Accuracy: 0.914

f1_score:  0.9132201391432682
\precision:  0.9158471549402659
ecall:  0.914

Classification Report:
              precision    recall  f1-score   support

     Culture     0.8947    0.9067    0.9007       150
     Finance     0.9173    0.8133    0.8622       150
     Medical     0.9299    0.9733    0.9511       150
    Politics     0.8393    0.9400    0.8868       150
    Religion     0.9643    0.8100    0.8804       100
      Sports     0.9615    1.0000    0.9804       150
        Tech     0.9200    0.9200    0.9200       150

    accuracy                         0.9140      1000
   macro avg     0.9182    0.9090    0.9117      1000
weighted avg     0.9158    0.9140    0.9132      1000



# Few Shot

## Prepare Prompt

In [ ]:
data['system_message'] = '''You are a language model tasked with classifying Arabic newspaper articles into one of the predefined editorial categories based solely on the article's main topic and content. Carefully read each article and assign only one of the following categories that best reflects its primary subject matter:
1. Culture
2. Finance
3. Medical
4. Politics
5. Religion
6. Sports
7. Tech

Example 1
Article content:
أكد الفنان عزت أبوعوف، رئيس مهرجان القاهرة السينمائي، إلغاء حفل ختام المهرجان الذي كان من المزمع أن يقام غداً، وذلك بسبب الظروف غير المستقرة التي تمر بها مصر حالياً.ونفى أبوعوف في تصريحات لـ"العربية.نت" سفر الضيوف الأجانب خوفاً مما تشهده مصر من مظاهرات، وما تردد عن أن هذا السفر قد تسبب في عدم إمكانية إقامة حفل الختام، مؤكداً أن وزير الثقافة دكتور محمد صابر عرب هو من أصدر قرار الإلغاء.وعن توزيع الجوائز التي كان من المزمع أن يتم في حفل الختام، شرح أنه سيقام مؤتمر صحافي سيكون بديلاً عن الحفل وسيتم من خلاله إعلان توزيع الجوائز والأفلام الفائزة في هذه الدورة.يُذكر أن حفل افتتاح المهرجان أيضاً تم في هدوء وبعيداً عن الصخب المعتاد، وذلك أيضاً بسبب الظروف نفسها التي تشهدها مصر.

Predicted Category: Culture

Example 2
Article content:
قال الرئيس التنفيذي للشركة السعودية للكهرباء زياد الشيحة، في مقابلة عبر الهاتف مع قناة "العربية"، إنه لأول مرة في تاريخ الشركة تراجع استهلاك المملكة في فترات الذروة. وأضاف الشيحة أن التراجع الذي حدث في استهلاك المملكة في 2016، مقارنة مع عام 2015، دفع الشركة لمراجعة السعات المطلوبة لدى دراسة المشاريع الجديدة. وأكد أن مسألة انخفاض الحمل الذروي لأول مرة في تاريخ الشركة عن العام جلعنا نراجع المحطات المستقبلية والتي ستكون بعقود شراء الطاقة، وهذا سيكون لمشاربع الإنتاج وتتم مراجعة السعات المطلوبة خاصة مع قلة الحمل الذروي في 2016. وستزود الشركة السعودية للكهرباء شركة زين السعودية بشبكة الألياف البصرية الممتدة لـ 60 ألف كم. وأوضح الشيحة أن "قطاع التوليد سيطرح للخصخصة كما هو معلن، ونعمل على الموضوع بشكل متوازن وشبه يومي". وكانت خسائر شركة السعودية للكهرباء قد تفاقمت بأكثر من 60%، في الربع الأخير من العام الماضي، مقارنة بالربع المماثل من عام 2015، لتبلغ 2.34 مليار ريال. من ناحية أخرى، ارتفعت أرباح الشركة بنسبة 37%، خلال العام الماضي، مقارنةً بعام 2015، لتبلغ 2.1 مليار ريال. وأرجعت الشركة تفاقم الخسائر الفصلية إلى ارتفاع تكلفة المبيعات نتيجة الزيادة في أسعار الوقود وارتفاع المصاريف التشغيلية.

Predicted Category: Finance

Example 3
Article content:
يعتقد بعض المدخنين أن السجائر الإلكترونية تعد أحد أهم العوامل المساعدة في الإقلاع عن التدخين، في حين يعتقد البعض الآخر أنها تعتبر وسيلة إغواء للاستمرار في الخضوع لتلك العادة المدمرة، إلا أن الأبحاث الطبية الحديثة تشير إلى أن الأخطار والفوائد لتلك النوعية من السجائر لاتزال غير معلومة بصورة واضحة بين المدخنين، وذلك وفق ما نشرت وكالة أنباء الشرق الأوسط المصرية. وكانت مجموعة من الباحثين قد أجرت أبحاثها على أكثر من 64 مدخنا، ولم ينجحوا في تحقيق إجماع حول الفوائد والأضرار المحتملة للسجائر الإلكترونية، وهو ما قد يعكس انقساما في المجتمع الطبي حول مدى ملاءمة تعزيز السجائر الإلكترونية كبديل أكثر أمنا للتدخين. وأوضح الباحثون أن معظم المشاركين في الدراسة يرون أن التدخين يعتبر شكلا من الإدمان، حيث تلعب الإرادة دورا قويا في الإقلاع عن هذه العادة المدمرة، في الوقت الذى حاول فيه جميع المشاركين في الدراسة مرة واحدة على الأقل الإقلاع عن العادة المدمرة.

Predicted Category: Medical

Example 4
Article content:
أعرب المتحدث باسم الهيئة العليا للمفاوضات السورية سالم المسلط عن أمله في أن تنتقل روسيا فعليا لتقف إلى جانب الشعب السوري بدلا من النظام، وذلك عقب قراره بسحب القوات الروسية من سوريا. وأضاف المسلط أن هناك جدية لمست مؤخرا حيال المواقف الروسية للدفع نحو الحل السياسي للأزمة في سوريا، خلال جولة المحادثات الجديدة التي انطلقت في جنيف. هذا وأعلن متحدث باسم الرئيس الروسي فلاديمير بوتين بوتين بأن روسيا أبلغت الأسد بقرار سحب الجزء الرئيسي من القوات الروسية من سوريا. وقال المتحدث إن بوتين خلال اجتماعه بوزير دفاعه أمر اعتبارا من اليوم (الثلاثاء) ببدء سحب الجزء الرئيسي من القوات الروسية. في حين قال متحدث باسم بوتين إن القاعدة البحرية والجوية الروسية في سوريا تستمر في العمل كما في السابق. ووفقا للكرملين فإن بوتين طلب من وزير خارجيته سيرغي لافروف تكثيف الدور الروسي في عملية السلام في سوريا، مشيرا إلى ان  القوات الروسية في سوريا أوجدت ظروفا ملائمة لعملية السلام.

Predicted Category: Politics

Example 5
Article content:
أكد عبداللطيف بخاري، رئيس لجنة الخبراء باتحاد القدم السعودي أن اتحاده يبحث عن 15 منصباً في اللجان الآسيوية، وأن هناك لجنة برئاسة خالد المرزوقي، عضو مجلس إدارة الاتحاد مكلفة بالاختيار. وقال بخاري لـ"في المرمى":" الترشيحات تتم بناء على معايير قارية، ونحن نبحث عن 15 منصباً في لجان الاتحاد الآسيوي". وبين بخاري أن لجنة المسابقات اقترحت إيجاد مراقب لكل مباراة، وزاد:" تقارير المراقب لن تغني عن تقارير الحكم، أما بالنسبة لتقارير الأول فيمكن للجنة الانضباط الاستناد عليها".

Predicted Category: Sports

Example 6
Article content:
يبدو أن هاتف #آيفون7  الجديد الذي ستصدره شركة آبل، لن يحمل الكثير من التغيرات، بحسب ما أفاد تقرير لـ "وول ستريت جورنول". فالهاتف الجديد سيأتي شبيها بالنسخة الحالية (آيفون6)، مع تغيير جذري على صعيد "الصوت" والسماعات. وحسب تقرير الصحيفة قد تزيل شركة #آبل منفذ سماعة الصوت، لتدمجه بالمنفذ الذي يوضع فيه شاحن الهاتف في الأسفل. وتراهن آبل من خلال إزالة المنفذ على جعل هاتفها الجديد "أرفع"، ومضاد للماء فإزالة فتحة السماعة، ستمنع تسرب الماء إلى الجهاز عبر الثقب، وتعطيله. ومن شأن إزالة المنفذ الذي يصل قطره إلى 2.5 ميلليمتر أن ينعكس إيجابا أيضاً على البطارية، على اعتبار أن التغيير سيفسح مساحة جديدة يمكن استغلالها. إلا أن التقرير لم يفصل كيف يمكن لمنفذ الشحن الجديد أن يمنع بدوره تسرب الماء إلى داخل الهاتف. في المقابل، يرى بعض منتقدي الشكل الجديد أو التغيير المنتظر أن الاستغناء عن السماعات التقليدية، سيجبر المستخدمين على شراء السماعات الأغلى التي تعمل بتقنية "بلوتوث".

Predicted Category: Tech

Example 7
Article content:
} قال رسول الله صلى الله عليه وسلم: «من أتى فراشه وهو ينوي أن يقوم يصلي من الليل فغلبته عينه حتى أصبح، كتب له ما نوى، وكان نومه صدقة عليه من ربه».} وقال صلى الله عليه وسلم: «ما تشاور قومإلا هداهم الله لأرشد أمورهم».

Predicted Category: Religion
'''
data['user_content'] = 'The article you need to classify\nArticle:\n' + data['content'] +'\nPredicted Category:'

data2 = data
data = data[['system_message','user_content']]

In [ ]:
data.to_excel('NC-FewShot-gpt.xlsx')

In [ ]:
# Convert to JSON String
data = data.to_json(orient='records')

# parse JSON string and convert it into a list of python dictionaries
datadict = json.loads(data)

## Format Prompt

In [ ]:
# format dict into shape for fine tuning
few_shot_data = format_prompt(datadict)

# Save as jsonl
convert_to_jsonl_and_save(few_shot_data, 'NC-FewShot.jsonl')

✅ Data successfully written to NC-FewShot.jsonl


## Predict

In [ ]:
response = client.chat.completions.create(
      model = model,
      messages= few_shot_data[66]['messages'],
      temperature = 0.2,
      max_tokens= 20
  )

response.choices[0].message.content

'Predicted Category: Finance'

In [ ]:
y_true = data2['subdirectory'].values

In [ ]:
fs = pd.DataFrame()

fs['text'] = data2['content']
fs['Label'] = y_true
fs = fs.reset_index(drop=True)
fs.head()

,text,Label
0,استقبل الوسط المسرحي الإماراتي فوز الإمارات بر...,Culture
1,استضاف مركز الشارقة لفن الخط العربي والزخرفة م...,Culture
2,باسمة يونس قد يبدو العنوان اسماً لرواية؛ لكنه ...,Culture
3,أبوظبي: «الخليج» أكد عدد من الخبراء والمسؤولين...,Culture
4,يمكن القول باطمئنان أن شهر رمضان المبارك هو شه...,Culture


In [ ]:
store_predictions(fs, model, few_shot_data)

fine tuned model id is : gpt-4.1-2025-04-14


In [ ]:
pred = pd.read_csv('NC-gpt41-FewShot-predictions-temp0.2.csv')

In [ ]:
pred['Prediction'].value_counts()

,count
Prediction,
Predicted Category: Politics,170
Predicted Category: Tech,153
Predicted Category: Medical,148
Predicted Category: Culture,141
Predicted Category: Finance,123
Predicted Category: Sports,103
Predicted Category: Religion,83
Sports,53
Culture,11


In [ ]:
pred['Prediction'].isnull().sum()

np.int64(0)

In [ ]:
import numpy as np

preds = []
for answer in pred['Prediction']:
    if (
          "الرياضة" in answer
          or "Sports" in answer
          or "sports" in answer
          or "sport" in answer
          or "Sport" in answer
      ):
      preds.append("Sports")

    elif (
        "الصحة" in answer
          or "الطب" in answer
          or "Medical" in answer
          or "medical" in answer
        ):
      preds.append("Medical")

    elif (
        "الثقافة" in answer
          or "Culture" in answer
          or "culture" in answer
        ):
      preds.append("Culture")

    elif (
        "المال" in answer
        or "Finance" in answer
        or "finance" in answer
        ):
      preds.append("Finance")

    elif (
        "السياسة" in answer
        or "Politics" in answer
        or "politics" in answer
        ):
      preds.append("Politics")

    elif (
        "الدين" in answer
        or "Religion" in answer
        or "religion" in answer
        ):
      preds.append("Religion")

    elif (
        "التكنولوجيا" in answer
        or "Tech" in answer
        or "Technology" in answer
        or "tech" in answer
        or "technology" in answer
        ):
      preds.append("Tech")

    else:
       preds.append("None")

In [ ]:
np.unique(preds)

array(['Culture', 'Finance', 'Medical', 'Politics', 'Religion', 'Sports',
       'Tech'], dtype='<U8')

In [ ]:
from sklearn.metrics import (f1_score,
                             precision_score,
                             recall_score,
                             classification_report,
                             confusion_matrix)

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score

labels = ['Culture', 'Finance', 'Medical', 'Politics',
          'Religion', 'Sports', 'Tech']
mapping = {'Culture':0, 'Finance':1, 'Medical':2, 'Politics':3,
           'Religion':4, 'Sports':5, 'Tech':6}
def map_func(x):
    return mapping.get(x, 1)

y_true = np.vectorize(map_func)(fs.Label)
y_pred = np.vectorize(map_func)(preds)

# Calculate accuracy
accuracy = accuracy_score(y_true=y_true, y_pred=y_pred)
print(f'Accuracy: {accuracy:.3f}')

# Generate accuracy report
unique_labels = set(y_true)  # Get unique labels

f1t = f1_score(y_true=y_true, y_pred=y_pred, average = 'weighted')
print('\nf1_score: ', f1t)

prec = precision_score(y_true=y_true, y_pred=y_pred, average = 'weighted')
print('\precision: ', prec)

recall = recall_score(y_true=y_true, y_pred=y_pred, average = 'weighted')
print('\recall: ', recall)

# Generate classification report
class_report = classification_report(y_true=y_true,
                                     y_pred=y_pred,
                                     digits = 4,
                                     target_names=labels)
print('\nClassification Report:')
print(class_report)

Accuracy: 0.919

f1_score:  0.9183187147864369
\precision:  0.9220437379387454
ecall:  0.919

Classification Report:
              precision    recall  f1-score   support

     Culture     0.8947    0.9067    0.9007       150
     Finance     0.9457    0.8133    0.8746       150
     Medical     0.9236    0.9667    0.9446       150
    Politics     0.8412    0.9533    0.8938       150
    Religion     0.9880    0.8200    0.8962       100
      Sports     0.9615    1.0000    0.9804       150
        Tech     0.9216    0.9400    0.9307       150

    accuracy                         0.9190      1000
   macro avg     0.9252    0.9143    0.9173      1000
weighted avg     0.9220    0.9190    0.9183      1000



# CoT

## Prepare Prompt

In [ ]:
data['system_message'] = '''You are a language model tasked with classifying Arabic newspaper articles into one of the predefined editorial categories based solely on the article's main topic and content. Carefully read each article and assign only one of the following categories that best reflects its primary subject matter:
1. Culture
2. Finance
3. Medical
4. Politics
5. Religion
6. Sports
7. Tech

Follow these steps when analyzing the content of each article:

Step 1: Comprehend the Core Content
Read the full article carefully. Identify the central event, issue, or message. This may be a news story, commentary, or report involving domains like politics, health, economy, sports, technology, or culture.

Step 2: Identify the Dominant Theme
Determine the primary topic or theme. Focus on the article’s content and recurring concepts. Ignore secondary topics or tangents that do not define the article’s overall focus.

Step 3: Match the Article to a Category
Select the single category that best corresponds to the dominant theme. Be precise: choose the most specific and directly related category. Do not rely on assumptions or background knowledge outside the text. If the article touches on multiple themes, choose the one most emphasized or most central to the article’s purpose.


Example 1
Article content:
أكد الفنان عزت أبوعوف، رئيس مهرجان القاهرة السينمائي، إلغاء حفل ختام المهرجان الذي كان من المزمع أن يقام غداً، وذلك بسبب الظروف غير المستقرة التي تمر بها مصر حالياً.ونفى أبوعوف في تصريحات لـ"العربية.نت" سفر الضيوف الأجانب خوفاً مما تشهده مصر من مظاهرات، وما تردد عن أن هذا السفر قد تسبب في عدم إمكانية إقامة حفل الختام، مؤكداً أن وزير الثقافة دكتور محمد صابر عرب هو من أصدر قرار الإلغاء.وعن توزيع الجوائز التي كان من المزمع أن يتم في حفل الختام، شرح أنه سيقام مؤتمر صحافي سيكون بديلاً عن الحفل وسيتم من خلاله إعلان توزيع الجوائز والأفلام الفائزة في هذه الدورة.يُذكر أن حفل افتتاح المهرجان أيضاً تم في هدوء وبعيداً عن الصخب المعتاد، وذلك أيضاً بسبب الظروف نفسها التي تشهدها مصر.

Thoughts:
- Main content: cancellation of the closing ceremony of the Cairo International Film Festival
- Main theme: cultural and artistic events
- Category Milan: Culture

Predicted Category: 'Culture'

Example 2
Article content:
قال الرئيس التنفيذي للشركة السعودية للكهرباء زياد الشيحة، في مقابلة عبر الهاتف مع قناة "العربية"، إنه لأول مرة في تاريخ الشركة تراجع استهلاك المملكة في فترات الذروة. وأضاف الشيحة أن التراجع الذي حدث في استهلاك المملكة في 2016، مقارنة مع عام 2015، دفع الشركة لمراجعة السعات المطلوبة لدى دراسة المشاريع الجديدة. وأكد أن مسألة انخفاض الحمل الذروي لأول مرة في تاريخ الشركة عن العام جلعنا نراجع المحطات المستقبلية والتي ستكون بعقود شراء الطاقة، وهذا سيكون لمشاربع الإنتاج وتتم مراجعة السعات المطلوبة خاصة مع قلة الحمل الذروي في 2016. وستزود الشركة السعودية للكهرباء شركة زين السعودية بشبكة الألياف البصرية الممتدة لـ 60 ألف كم. وأوضح الشيحة أن "قطاع التوليد سيطرح للخصخصة كما هو معلن، ونعمل على الموضوع بشكل متوازن وشبه يومي". وكانت خسائر شركة السعودية للكهرباء قد تفاقمت بأكثر من 60%، في الربع الأخير من العام الماضي، مقارنة بالربع المماثل من عام 2015، لتبلغ 2.34 مليار ريال. من ناحية أخرى، ارتفعت أرباح الشركة بنسبة 37%، خلال العام الماضي، مقارنةً بعام 2015، لتبلغ 2.1 مليار ريال. وأرجعت الشركة تفاقم الخسائر الفصلية إلى ارتفاع تكلفة المبيعات نتيجة الزيادة في أسعار الوقود وارتفاع المصاريف التشغيلية.

Thoughts:
- Main content: interview with Ziyad Al-Shiha, CEO of the Saudi Electricity Company
- Main theme: Economy and energy
- Category Milan: Finance

Predicted category: 'Finance'

Example 3
Article content:
يعتقد بعض المدخنين أن السجائر الإلكترونية تعد أحد أهم العوامل المساعدة في الإقلاع عن التدخين، في حين يعتقد البعض الآخر أنها تعتبر وسيلة إغواء للاستمرار في الخضوع لتلك العادة المدمرة، إلا أن الأبحاث الطبية الحديثة تشير إلى أن الأخطار والفوائد لتلك النوعية من السجائر لاتزال غير معلومة بصورة واضحة بين المدخنين، وذلك وفق ما نشرت وكالة أنباء الشرق الأوسط المصرية. وكانت مجموعة من الباحثين قد أجرت أبحاثها على أكثر من 64 مدخنا، ولم ينجحوا في تحقيق إجماع حول الفوائد والأضرار المحتملة للسجائر الإلكترونية، وهو ما قد يعكس انقساما في المجتمع الطبي حول مدى ملاءمة تعزيز السجائر الإلكترونية كبديل أكثر أمنا للتدخين. وأوضح الباحثون أن معظم المشاركين في الدراسة يرون أن التدخين يعتبر شكلا من الإدمان، حيث تلعب الإرادة دورا قويا في الإقلاع عن هذه العادة المدمرة، في الوقت الذى حاول فيه جميع المشاركين في الدراسة مرة واحدة على الأقل الإقلاع عن العادة المدمرة.

Thoughts:
- Main Content: electronic cigarettes and their role in quitting smoking
- Main Theme: Health and medicine
- Category Match: Medical

Predicted Category: ' Medical'

Example 4
Article content:
أعرب المتحدث باسم الهيئة العليا للمفاوضات السورية سالم المسلط عن أمله في أن تنتقل روسيا فعليا لتقف إلى جانب الشعب السوري بدلا من النظام، وذلك عقب قراره بسحب القوات الروسية من سوريا. وأضاف المسلط أن هناك جدية لمست مؤخرا حيال المواقف الروسية للدفع نحو الحل السياسي للأزمة في سوريا، خلال جولة المحادثات الجديدة التي انطلقت في جنيف. هذا وأعلن متحدث باسم الرئيس الروسي فلاديمير بوتين بوتين بأن روسيا أبلغت الأسد بقرار سحب الجزء الرئيسي من القوات الروسية من سوريا. وقال المتحدث إن بوتين خلال اجتماعه بوزير دفاعه أمر اعتبارا من اليوم (الثلاثاء) ببدء سحب الجزء الرئيسي من القوات الروسية. في حين قال متحدث باسم بوتين إن القاعدة البحرية والجوية الروسية في سوريا تستمر في العمل كما في السابق. ووفقا للكرملين فإن بوتين طلب من وزير خارجيته سيرغي لافروف تكثيف الدور الروسي في عملية السلام في سوريا، مشيرا إلى ان  القوات الروسية في سوريا أوجدت ظروفا ملائمة لعملية السلام.

Thoughts:
- Main content: statements from Salem Al-Muslat, spokesperson for the Syrian High Negotiations Committee
- Main theme: Politics and international relations
- Category Milan: Politics

Predicted Category: ' Politics'

Example 5
Article content:
أكد عبداللطيف بخاري، رئيس لجنة الخبراء باتحاد القدم السعودي أن اتحاده يبحث عن 15 منصباً في اللجان الآسيوية، وأن هناك لجنة برئاسة خالد المرزوقي، عضو مجلس إدارة الاتحاد مكلفة بالاختيار. وقال بخاري لـ"في المرمى":" الترشيحات تتم بناء على معايير قارية، ونحن نبحث عن 15 منصباً في لجان الاتحاد الآسيوي". وبين بخاري أن لجنة المسابقات اقترحت إيجاد مراقب لكل مباراة، وزاد:" تقارير المراقب لن تغني عن تقارير الحكم، أما بالنسبة لتقارير الأول فيمكن للجنة الانضباط الاستناد عليها".

Thoughts:
- Main content: statements by Abdul Latif Bukhari, head of the Experts Committee at the Saudi Football Federation
- Main theme: Sports
- Category Milan: Sports

Predicted Category: ' Sports'

Example 6
Article content:
يبدو أن هاتف #آيفون7  الجديد الذي ستصدره شركة آبل، لن يحمل الكثير من التغيرات، بحسب ما أفاد تقرير لـ "وول ستريت جورنول". فالهاتف الجديد سيأتي شبيها بالنسخة الحالية (آيفون6)، مع تغيير جذري على صعيد "الصوت" والسماعات. وحسب تقرير الصحيفة قد تزيل شركة #آبل منفذ سماعة الصوت، لتدمجه بالمنفذ الذي يوضع فيه شاحن الهاتف في الأسفل. وتراهن آبل من خلال إزالة المنفذ على جعل هاتفها الجديد "أرفع"، ومضاد للماء فإزالة فتحة السماعة، ستمنع تسرب الماء إلى الجهاز عبر الثقب، وتعطيله. ومن شأن إزالة المنفذ الذي يصل قطره إلى 2.5 ميلليمتر أن ينعكس إيجابا أيضاً على البطارية، على اعتبار أن التغيير سيفسح مساحة جديدة يمكن استغلالها. إلا أن التقرير لم يفصل كيف يمكن لمنفذ الشحن الجديد أن يمنع بدوره تسرب الماء إلى داخل الهاتف. في المقابل، يرى بعض منتقدي الشكل الجديد أو التغيير المنتظر أن الاستغناء عن السماعات التقليدية، سيجبر المستخدمين على شراء السماعات الأغلى التي تعمل بتقنية "بلوتوث".

Thoughts:
- Main content: a report on Apple’s upcoming iPhone 7
- Main theme: Technology
- Category Milan: Technology

Predicted Category: ' Tech'

Example 7
Article content:
} قال رسول الله صلى الله عليه وسلم: «من أتى فراشه وهو ينوي أن يقوم يصلي من الليل فغلبته عينه حتى أصبح، كتب له ما نوى، وكان نومه صدقة عليه من ربه».} وقال صلى الله عليه وسلم: «ما تشاور قومإلا هداهم الله لأرشد أمورهم».

Thoughts:
- Main content: Hadiths (sayings of the Prophet Muhammad peace be upon him)
- Main theme: Religion and creed
- Category Milan: Religion

Predicted Category: ' Religion'
'''
data['user_content'] = 'The article you need to classify\nArticle:\n' + data['content']

data2 = data
data = data[['system_message','user_content']]

In [ ]:
data.to_excel('NC-CoT-gpt.xlsx')

In [ ]:
# Convert to JSON String
data = data.to_json(orient='records')

# parse JSON string and convert it into a list of python dictionaries
datadict = json.loads(data)

## Format Prompt

In [ ]:
# format dict into shape for fine tuning
CoT_data = format_prompt(datadict)

# Save as jsonl
convert_to_jsonl_and_save(CoT_data, 'SA-CoT.jsonl')

✅ Data successfully written to SA-CoT.jsonl


## Predict

In [ ]:
response = client.chat.completions.create(
      model = model,
      messages= CoT_data[99]['messages'],
      temperature = 0.2,
      max_tokens= 512
  )

response.choices[0].message.content

'Step 1: Comprehend the Core Content  \nThe article discusses the performance of various stock markets in the region during a recent trading session. It details the positive and negative movements in the Saudi, Kuwaiti, Qatari, Bahraini, Omani, and Jordanian stock markets, referencing financial results, economic conditions, and investor sentiment.\n\nStep 2: Identify the Dominant Theme  \nThe dominant theme is the analysis and reporting of financial market performance, specifically stock exchanges in several countries, with a focus on financial results and economic factors affecting these markets.\n\nStep 3: Match the Article to a Category  \nThe article is primarily about financial markets and economic performance.\n\nPredicted Category: Finance'

In [ ]:
def predict(test, model):
  response = client.chat.completions.create(
      model = model,
      messages= test,
      temperature = 0.2,
      max_tokens= 512
  )
  return response.choices[0].message.content

In [ ]:
def store_predictions(test_df, model, test_data):
  print("fine tuned model id is :", model)
  test_df['Prediction']= None

  for index, row in test_df.iterrows():
    test_message = test_data[index]['messages']
    prediction_result = predict(test_message, model)
    test_df.at[index, 'Prediction'] = prediction_result

  test_df.to_csv("NC-GPT41-CoT-predictions-temp0.2.csv")

In [ ]:
y_true = data2['subdirectory'].values

In [ ]:
cot = pd.DataFrame()

cot['text'] = data2['content']
cot['Label'] = y_true
cot = cot.reset_index(drop=True)
cot.head()

,text,Label
0,استقبل الوسط المسرحي الإماراتي فوز الإمارات بر...,Culture
1,استضاف مركز الشارقة لفن الخط العربي والزخرفة م...,Culture
2,باسمة يونس قد يبدو العنوان اسماً لرواية؛ لكنه ...,Culture
3,أبوظبي: «الخليج» أكد عدد من الخبراء والمسؤولين...,Culture
4,يمكن القول باطمئنان أن شهر رمضان المبارك هو شه...,Culture


In [ ]:
cot.iloc[830:]

,text,Label,Prediction
830,المساء\nأعلن حزب العدالة والتنمية مقاطعته لمس...,Politics,NaN
831,أخبارنا المغربية ــ الرباط\nاعترف عبدالإله ابن...,Politics,NaN
832,أخبارنا المغربية \nتولى لحسن السكوري منصب وزار...,Politics,NaN
833,ذكرت صحيفة الاتحاد الاشتراكي في عددها الصادر ا...,Politics,NaN
834,أخبارنا المغربية : الرباط\nمعركة جديدة تلك الت...,Politics,NaN
...,...,...,...
995,كشفت شركة “سامسونج” الكورية الجنوبية، اليوم، ع...,Tech,NaN
996,"المريض ""المُنْحَبِس"" المصاب بشلل تام قد يكون ف...",Tech,NaN
997,يبدو أن مفهوم الحرب بمعناه المتعارف عليه سيتغي...,Tech,NaN
998,"تشتهر شركة ""ابل"" بجودة اصداراتها حيث انها أهم ...",Tech,NaN


In [ ]:
store_predictions(cot, model, CoT_data)

fine tuned model id is : gpt-4.1-2025-04-14


In [ ]:
pred = pd.read_csv('NC-GPT41-CoT-predictions-temp0.2.csv')

In [ ]:
pred['Prediction'].value_counts()

,count
Prediction,
Predicted Category: Sports,38
Predicted Category: Finance,20
Predicted Category: Medical,17
Predicted Category: 'Tech',15
Predicted Category: Politics,15
...,...
"Step 1: Comprehend the Core Content \nThe article discusses the Sheikh Zayed Grand Mosque Center’s adoption of the ""smart government"" initiative, launching an integrated system of services accessible via mobile phones. The article details various smart applications and digital services provided to visitors and users, such as booking tours, obtaining filming permits, accessing the library, and communicating with staff. It also mentions the center’s participation in GITEX Technology Week and emphasizes the center’s commitment to technological advancement to enhance its cultural mission.\n\nStep 2: Identify the Dominant Theme \nWhile the article references the mosque’s cultural and religious significance, the main focus is on the implementation and use of new technological solutions and smart services. The central event is the digital transformation of the center and its participation in a major technology exhibition (GITEX). The recurring concepts are technology, digital services, smart applications, and innovation in service delivery.\n\nStep 3: Match the Article to a Category \nAlthough there are cultural and religious undertones, the dominant theme is the adoption and application of technology in service delivery and institutional development.\n\nPredicted Category: Tech",1
"Step 1: Comprehend the Core Content \nThe article discusses the launch of three new servers by Fujitsu in the global market, specifically within the PRIMERGY BX900 blade server systems. It details the technical features, including Intel Xeon processors, and highlights the flexibility, efficiency, and advanced infrastructure these servers provide. The article also mentions the benefits for customers, such as reduced operational costs and adaptability for various IT needs.\n\nStep 2: Identify the Dominant Theme \nThe dominant theme is the introduction of new technology products (servers) and their technical specifications and benefits for IT infrastructure.\n\nStep 3: Match the Article to a Category \nThe article is primarily focused on technology, specifically new server hardware and IT solutions.\n\nPredicted Category: Tech",1
"Step 1: Comprehend the Core Content \nThe article reports on Brother Gulf International's completion of a business planning workshop for its major authorized distributors in the region. The event included discussions on increasing market share, profit margins, and growth in the multi-function device sector. The company honored key partners for their sales achievements in printers and multi-function devices, discussed future strategies focusing on advanced printers and devices, and announced marketing and incentive plans.\n\nStep 2: Identify the Dominant Theme \nThe main focus is on business strategies, distributor partnerships, sales achievements, and marketing plans related to technology products (printers and multi-function devices). The article centers on the technology sector, specifically the business and marketing aspects of tech hardware.\n\nStep 3: Match the Article to a Category \nWhile there are elements of finance (sales, profits), the dominant theme is the technology industry—specifically, the business of distributing and marketing tech devices.\n\nPredicted Category: Tech",1


In [ ]:
pred['Prediction'].isnull().sum()

np.int64(0)

In [ ]:
import numpy as np

def get_text_after_sentiment(row):
    text = row['Prediction']
    word = 'Predicted Category:'
    idx = text.find(word)
    if idx != -1:
        return text[idx + len(word):].strip()
    return np.nan

# Apply to create new column
pred['Normalized Output'] = pred.apply(get_text_after_sentiment, axis=1)

pred.head()

,text,Label,Prediction,Normalized Output
0,استقبل الوسط المسرحي الإماراتي فوز الإمارات بر...,Culture,Step 1: Comprehend the Core Content \nThe art...,Culture
1,استضاف مركز الشارقة لفن الخط العربي والزخرفة م...,Culture,Step 1: Comprehend the Core Content \nThe art...,Culture
2,باسمة يونس قد يبدو العنوان اسماً لرواية؛ لكنه ...,Culture,Step 1: Comprehend the Core Content \nThe art...,Culture
3,أبوظبي: «الخليج» أكد عدد من الخبراء والمسؤولين...,Culture,Step 1: Comprehend the Core Content \nThe art...,Culture
4,يمكن القول باطمئنان أن شهر رمضان المبارك هو شه...,Culture,Step 1: Comprehend the Core Content \nThe art...,Culture


In [ ]:
pred['Normalized Output'].value_counts()

,count
Normalized Output,
Culture,148
Medical,145
Sports,141
Politics,136
Tech,132
...,...
"Sports\n\nThoughts:\n- Main content: The article discusses the involvement of Moroccan producer RedOne with Real Madrid after their Champions League victory, including his presence with the team and his role in producing the club's official anthem.\n- Main theme: The focus is on Real Madrid's football victory, the celebration, and RedOne's connection to the team in a sports context.\n- Category Match: Sports",1
"Sports\n\nThoughts:\n- Main content: Interview with French coach Claude Le Roy about the Moroccan national football team, his wish to coach them, and his predictions for their success in upcoming tournaments.\n- Main theme: Football (soccer), coaching, and sports competition.\n- Category match: Sports",1
"Sports\n\nThoughts:\n- Main content: Clarification by the Moroccan Football Federation regarding the call-up of Cameroonian-origin player Ryan Mmaee to the Moroccan national team, including statements from the national coach and the federation about the player's willingness and the procedures for national team selection.\n- Main theme: National football team selection and related administrative/athletic procedures.\n- Category Match: Sports",1


In [ ]:
pred['Normalized Output'].isnull().sum()

np.int64(0)

In [ ]:
import numpy as np

preds = []
for answer in pred['Prediction']:
    if (
          "الرياضة" in answer
          or "Sports" in answer
          or "sports" in answer
          or "sport" in answer
          or "Sport" in answer
      ):
      preds.append("Sports")

    elif (
        "الصحة" in answer
          or "الطب" in answer
          or "Medical" in answer
          or "medical" in answer
        ):
      preds.append("Medical")

    elif (
        "الثقافة" in answer
          or "Culture" in answer
          or "culture" in answer
        ):
      preds.append("Culture")

    elif (
        "المال" in answer
        or "Finance" in answer
        or "finance" in answer
        ):
      preds.append("Finance")

    elif (
        "السياسة" in answer
        or "Politics" in answer
        or "politics" in answer
        ):
      preds.append("Politics")

    elif (
        "الدين" in answer
        or "Religion" in answer
        or "religion" in answer
        ):
      preds.append("Religion")

    elif (
        "التكنولوجيا" in answer
        or "Tech" in answer
        or "Technology" in answer
        or "tech" in answer
        or "technology" in answer
        ):
      preds.append("Tech")

    else:
      print(answer)
      preds.append("None")

In [ ]:
np.unique(preds)

array(['Culture', 'Finance', 'Medical', 'Politics', 'Religion', 'Sports',
       'Tech'], dtype='<U8')

In [ ]:
from sklearn.metrics import (f1_score,
                             precision_score,
                             recall_score,
                             classification_report,
                             confusion_matrix)

In [ ]:
len(y_pred)

170

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score

labels = ['Culture', 'Finance', 'Medical', 'Politics',
          'Religion', 'Sports', 'Tech']
mapping = {'Culture':0, 'Finance':1, 'Medical':2, 'Politics':3,
           'Religion':4, 'Sports':5, 'Tech':6}
def map_func(x):
    return mapping.get(x, 1)

y_true = np.vectorize(map_func)(cot.Label)
y_pred = np.vectorize(map_func)(preds)

# Calculate accuracy
accuracy = accuracy_score(y_true=y_true, y_pred=y_pred)
print(f'Accuracy: {accuracy:.3f}')

# Generate accuracy report
unique_labels = set(y_true)  # Get unique labels

f1t = f1_score(y_true=y_true, y_pred=y_pred, average = 'weighted')
print('\nf1_score: ', f1t)

prec = precision_score(y_true=y_true, y_pred=y_pred, average = 'weighted')
print('\precision: ', prec)

recall = recall_score(y_true=y_true, y_pred=y_pred, average = 'weighted')
print('\recall: ', recall)

# Generate classification report
class_report = classification_report(y_true=y_true,
                                     y_pred=y_pred,
                                     digits = 4,
                                     target_names=labels)
print('\nClassification Report:')
print(class_report)

Accuracy: 0.865

f1_score:  0.8645558482663196
\precision:  0.8737335262798648
ecall:  0.865

Classification Report:
              precision    recall  f1-score   support

     Culture     0.8649    0.8533    0.8591       150
     Finance     0.8667    0.7800    0.8211       150
     Medical     0.8820    0.9467    0.9132       150
    Politics     0.8857    0.8267    0.8552       150
    Religion     0.9620    0.7600    0.8492       100
      Sports     0.7500    1.0000    0.8571       150
        Tech     0.9343    0.8533    0.8920       150

    accuracy                         0.8650      1000
   macro avg     0.8779    0.8600    0.8638      1000
weighted avg     0.8737    0.8650    0.8646      1000

